In this notebook, the default nside for both beam and sky is 512.

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../MERS/')

from fg_model import *
from SEDfitting import *

In [ ]:
# Set up the frequency list: [50, 200] MHz
freq_list = np.arange(50, 201, 2)
np.shape(freq_list)

In [ ]:

GSM_sky = GSM_maps(freq_list, beam_transfer=None)

mel_diffuse, pivot_specidx_map=CNN_PL_sky(freq_list, beam_transfer=None, nside=128, return_spec_index=True)

# hp.mollview(mel_sync[:,0], norm='hist', title='Mel_sync (50 MHz)')
# plt.show()

# hp.mollview(mel_ff[:,0], norm='hist', title='Mel_ff (50 MHz)')
# plt.show()

# plt.plot(freq_list, mel_diffuse[0,:])
# plt.xlabel('Frequency (MHz)')
# plt.ylabel('Temperature (K)')
# plt.title('Mel_diffuse (pixel 0)')
# plt.show()

# hp.mollview(mel_diffuse[:,0], norm='hist', title='Mel_diffuse (50 MHz)')
# plt.show()


# hp.mollview(GSM_sky[:,0], norm='hist', title='GSM Sky (50MHZ)')
# plt.show()

# Delete Mel_sync_model and Mel_ff_model


# 1. Fitting CNN-PL with fixed pivot values 

In [ ]:
# Mel_moment_coeffs_maps_4_fixed, Mel_loss_map_4_fixed = fit_fg_cube(mel_diffuse, pivot_specidx_map, freq_list, nu_ref=None, max_order=4, adaptive_pivot=False)
# Mel_moment_coeffs_maps_5_fixed, Mel_loss_map_5_fixed = fit_fg_cube(mel_diffuse, pivot_specidx_map, freq_list, nu_ref=None, max_order=5, adaptive_pivot=False)
# Mel_moment_coeffs_maps_6_fixed, Mel_loss_map_6_fixed = fit_fg_cube(mel_diffuse, pivot_specidx_map, freq_list, nu_ref=None, max_order=6, adaptive_pivot=False)
# Mel_moment_coeffs_maps_7_fixed, Mel_loss_map_7_fixed = fit_fg_cube(mel_diffuse, pivot_specidx_map, freq_list, nu_ref=None, max_order=7, adaptive_pivot=False)
Mel_moment_coeffs_maps_8_fixed, Mel_loss_map_8_fixed = fit_fg_cube(mel_diffuse, pivot_specidx_map, freq_list, nu_ref=None, max_order=8, adaptive_pivot=False)

# 2. Fitting GSM with fixed pivot values 

In [ ]:
GSM_coeffs_maps_4, GSM_loss_map_4 = fit_entire_fg_map(GSM_sky, pivot_specidx_map, freq_list, nu_ref=None, max_order=4, fixed_pivot=True)

In [ ]:
GSM_coeffs_maps_5, GSM_loss_map_5 = fit_entire_fg_map(GSM_sky, pivot_specidx_map, freq_list, nu_ref=None, max_order=5, fixed_pivot=True)

In [ ]:
GSM_coeffs_maps_6, GSM_loss_map_6 = fit_entire_fg_map(GSM_sky, pivot_specidx_map, freq_list, nu_ref=None, max_order=6, fixed_pivot=True)

## Visualisations

In [ ]:
hp.mollview(Mel_loss_map_5_adapted, title='Ratio of loss')

### Visualise the residual error maps

In [ ]:

# Generate a figure with 6 subplots, with the layout as 3 rows and 2 columns.
# The first row are healmaps of Mel_loss_map_3 and GSM_loss_map_3, with the same colorbar.
# The second row are healmaps of Mel_loss_map_4 and GSM_loss_map_4, with the same colorbar.
# The third row are healmaps of Mel_loss_map_5 and GSM_loss_map_5, with the same colorbar.

def plot_loss_comparison(savepath=None):
    plt.figure(figsize=(12, 12))
    
    maps = [
        (np.log10(Mel_loss_map_4_fixed), np.log10(GSM_loss_map_4), '4 Modes'), 
        (np.log10(Mel_loss_map_5_fixed), np.log10(GSM_loss_map_5), '5 Modes'),
        (np.log10(Mel_loss_map_6_fixed), np.log10(GSM_loss_map_6), '6 Modes')
    ]
    titles = ['CNN-PL', 'GSM', '']
    
    gs = plt.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0.01)
    
    for row in range(3):
        map1, map2, label = maps[row]
        
        # vmin = min(np.nanpercentile(map1, 1), np.nanpercentile(map2, 1))
        # vmax = max(np.nanpercentile(map1, 99), np.nanpercentile(map2, 99))
        vmin = -7
        vmax = -4
        
        cbar='bwr'
        ax1 = plt.subplot(gs[row, 0], projection='mollweide')
        im1 = hp.mollview(map1, title='', hold=True, 
                          min=vmin, max=vmax, 
                          #norm='log',
                          cmap=cbar)
        plt.text(0.5, 1.1, f'{titles[0]} {label}', 
                ha='center', va='bottom', transform=ax1.transAxes, fontsize=12)
        
        ax2 = plt.subplot(gs[row, 1], projection='mollweide')
        im2 = hp.mollview(map2, title='', hold=True, 
                          min=vmin, max=vmax, 
                          #norm='log',
                          cmap=cbar)
        plt.text(0.5, 1.1, f'{titles[1]} {label}', 
                ha='center', va='bottom', transform=ax2.transAxes, fontsize=12)
        
        # cax = plt.subplot(gs[row, 2])
        # plt.colorbar(im1['cmap'], cax=cax).set_label('Loss Value', fontsize=10)
    if savepath is not None:
        plt.savefig(savepath, dpi=300, bbox_inches='tight')
    #plt.suptitle('Spectral Moment Fitting Error Comparison', y=0.93, fontsize=14)
    plt.show()

plot_loss_comparison(savepath='../outputs/loss_comparison.pdf')

### Visualise the moment coefficient maps (N=6)

In [ ]:

def plot_moment_maps(savepath=None):
    plt.figure(figsize=(16, 16))
    

    titles = [['CNN-PL']*6, ['GSM']*6]
    
    gs = plt.GridSpec(4, 3, width_ratios=[1, 1, 1], height_ratios=[1,1,1,1], wspace=0.01)
    
    # Get the 16 and 84 percentiles
    # vmin = np.percentile(Mel_moment_coeffs_maps_6_fixed, 16)
    # vmax = np.percentile(Mel_moment_coeffs_maps_6_fixed, 84)

    vmin_ls = []
    vmax_ls = []

    for i in range(6):
        vmin_Mel = np.percentile(Mel_moment_coeffs_maps_6_fixed[:,i], 16)
        vmax_Mel = np.percentile(Mel_moment_coeffs_maps_6_fixed[:,i], 84)
        vmin_GSM = np.percentile(GSM_coeffs_maps_6[:,i], 16)
        vmax_GSM = np.percentile(GSM_coeffs_maps_6[:,i], 84)
        vmin_ls.append(min(vmin_Mel, vmin_GSM))
        vmax_ls.append(max(vmax_Mel, vmax_GSM))
        

    for row in range(2):
        for col in range(3):
            ind=row*3+col
            map=Mel_moment_coeffs_maps_6_fixed[:,ind]
            # vmin = vmin_ls[ind]
            # vmax = vmax_ls[ind]
            vmin = np.percentile(map, 16)
            vmax = np.percentile(map, 84)
        
            if ind==0:
                # not using diverging colormap
                cbar='bwr'
            else:
                # using diverging colormap
                cbar='bwr'
            ax1 = plt.subplot(gs[row, col], projection='mollweide')
            im1 = hp.mollview(map, title='', hold=True, 
                            min=vmin, max=vmax, 
                            #norm='log',
                            cmap=cbar)
            plt.text(0.5, 1.1, f'{titles[0][ind]} Mode {ind}', 
                    ha='center', va='bottom', transform=ax1.transAxes, fontsize=14)

    for row in range(2):
        for col in range(3):
            ind=row*3+col
            map=GSM_coeffs_maps_6[:,ind]

            vmin = np.percentile(map, 16)
            vmax = np.percentile(map, 84)
            # vmin = vmin_ls[ind]
            # vmax = vmax_ls[ind]
        
            if ind==0:
                # not using diverging colormap
                cbar='bwr'
            else:
                # using diverging colormap
                cbar='bwr'
            ax1 = plt.subplot(gs[row+2, col], projection='mollweide')
            im1 = hp.mollview(map, title='', hold=True, 
                            min=vmin, max=vmax, 
                            #norm='log',
                            cmap=cbar)
            plt.text(0.5, 1.1, f'{titles[1][ind]} Mode {ind}', 
                    ha='center', va='bottom', transform=ax1.transAxes, fontsize=14)

    if savepath is not None:
        plt.savefig(savepath, dpi=300, bbox_inches='tight')
    #plt.suptitle('Spectral Moment Fitting Error Comparison', y=0.93, fontsize=14)
    plt.show()

plot_moment_maps(savepath='../outputs/moment_maps_CNNPL_GSM.pdf')